# Prepare Calendar Table

Generates a comprehensive date-dimension table (`databricks_rg.silver.calendar`) spanning **1940-01-01 to 2030-12-31**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Generate date sequence from 1940-01-01 to 2030-12-31
start_date = "1940-01-01"
end_date = "2030-12-31"

df = (
    spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date")
)

calendar_df = df.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),
    F.year("date").alias("year"),
    F.month("date").alias("month"),
    F.dayofmonth("date").alias("day"),
    F.dayofweek("date").alias("day_of_week"),          # 1=Sun … 7=Sat
    F.dayofyear("date").alias("day_of_year"),
    F.weekofyear("date").alias("week_of_year"),
    F.quarter("date").alias("quarter"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_short_name"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_short_name"),
    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")).alias("year_quarter"),
    F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("date").isin(1, 7), False).otherwise(True).alias("is_weekday"),
    F.trunc("date", "month").alias("first_day_of_month"),
    F.last_day("date").alias("last_day_of_month"),
    F.trunc("date", "quarter").alias("first_day_of_quarter"),
    F.date_add(F.add_months(F.trunc("date", "quarter"), 3), -1).alias("last_day_of_quarter"),
    F.trunc("date", "year").alias("first_day_of_year"),
    F.to_date(F.concat(F.year("date"), F.lit("-12-31"))).alias("last_day_of_year"),
    F.dayofmonth(F.last_day("date")).alias("days_in_month"),
    F.when(
        (F.year("date") % 400 == 0) |
        ((F.year("date") % 4 == 0) & (F.year("date") % 100 != 0)),
        True
    ).otherwise(False).alias("is_leap_year"),
    F.ceil(F.month("date") / 6).alias("half_year"),          # 1 = H1, 2 = H2
    F.ceil(F.month("date") / 4).alias("tertile"),             # 1-3 (four-month periods)
)

# Write to Unity Catalog
calendar_df.write.mode("overwrite").saveAsTable("databricks_rg.silver.calendar")

print(f"Calendar table written — {calendar_df.count():,} rows")
display(spark.table("databricks_rg.silver.calendar").limit(10))